# BT4012 Kaggle 2026: illicit Bitcoin transactions, second submission
**Student:** e1122390 · **Public leaderboard ROC-AUC:** 0.95441

This notebook reproduces my second Kaggle submission file, `Submission/submission_2_e1122390.csv`. It is a single
XGBoost model, averaged over 10 random seeds, on 44 selected features, with no stacking or ensemble. It runs in
about a minute. My main submission is `e1122390.ipynb`.

**Steps**
1. Load the data.
2. Build graph features, one graph per time step.
3. Define the model: score every feature on how much it changed over time and how useful it is, keep it, convert
   it to a rank or drop it, keep the top 44, then train XGBoost.
4. Train on steps 1–35 and write the submission.
5. Time-based validation (train on steps 1–21, evaluate on 22–35): ROC-AUC, precision, recall and F1.
   Setting `RUN_TIME_CHECK = False` skips this; the submission does not depend on it.

## 1. Load the data
`label` is 1 for illicit, 0 for licit (legitimate) and empty for unlabelled transactions. Only labelled training
transactions are used to train the model. Unlabelled and test transactions are still used for the graph features,
which need no labels.

In [1]:
# Load the competition files.
from pathlib import Path

import lightgbm as lgb
import networkx as nx
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (accuracy_score, average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from IPython.display import display

SEED = 42
RUN_TIME_CHECK = True   # False skips the time-based validation (section 5); submission unchanged

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "train.csv").exists():  # allow running from Submission/
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"            # put the competition CSVs here (see README)
OUT_DIR = ROOT / "Submission"
assert (DATA_DIR / "train.csv").exists(), "Put the competition CSVs in data/ (see README)."
OUT_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
edges = pd.read_csv(DATA_DIR / "txs_edgelist.csv")
FEAT_COLS = [c for c in train.columns if c.startswith("feat_")]
print(f"train {train.shape}, test {test.shape}, edges {len(edges):,}")

train (141772, 168), test (15329, 168), edges 234,355


## 2. Graph features (one graph per time step)
Every link joins two transactions in the same time step, so I build one graph per step. The seven features are
in-degree, out-degree and total degree (number of connections), PageRank (importance within the network),
clustering coefficient (how closely neighbours are connected), connected-component size (size of the connected
group) and k-core number (how strongly connected the surrounding group is). Links with an end that has no
features are left out here. No labels are used. None of these features make the top 44 in this model.

In [2]:
# Build seven graph features per transaction, one graph per time step.
def graph_features(tx_steps, edges):
    """Seven graph features per transaction, one graph per time step. No labels are used."""
    step_of = dict(zip(tx_steps["txId"], tx_steps["time_step"]))
    e = edges.copy()
    e["ts1"] = e["txId1"].map(step_of)
    e["ts2"] = e["txId2"].map(step_of)
    e = e.dropna(subset=["ts1", "ts2"])  # keep edges whose two ends both have features
    e["time_step"] = e["ts1"].astype(int)

    rows = []
    for ts, g_edges in e.groupby("time_step"):
        ids = tx_steps.loc[tx_steps["time_step"] == ts, "txId"]
        G = nx.DiGraph()
        G.add_nodes_from(ids)
        G.add_edges_from(g_edges[["txId1", "txId2"]].itertuples(index=False, name=None))
        Gu = G.to_undirected()
        pagerank = nx.pagerank(G, alpha=0.85, max_iter=100) if G.number_of_edges() else {n: 0.0 for n in G}
        clustering = nx.clustering(Gu)
        comp_size = {n: len(comp) for comp in nx.connected_components(Gu) for n in comp}
        Gu.remove_edges_from(nx.selfloop_edges(Gu))  # core_number does not accept self-loops
        core_num = nx.core_number(Gu)

        d = pd.DataFrame({"txId": list(ids)})
        d["graph_in_degree"] = d["txId"].map(dict(G.in_degree())).fillna(0)  # links coming in
        d["graph_out_degree"] = d["txId"].map(dict(G.out_degree())).fillna(0)  # links going out
        d["graph_degree"] = d["txId"].map(dict(Gu.degree())).fillna(0)  # number of connections
        d["graph_pagerank"] = d["txId"].map(pagerank).fillna(0)  # importance within the network
        d["graph_clustering"] = d["txId"].map(clustering).fillna(0)  # how closely its neighbours are connected
        d["graph_component_size"] = d["txId"].map(comp_size).fillna(1)  # size of its connected group
        d["graph_core_number"] = d["txId"].map(core_num).fillna(0)  # how strongly connected its group is
        rows.append(d)
    return pd.concat(rows, ignore_index=True)


# every transaction with features (train + test), with its time step
tx_steps = (pd.concat([train[["txId", "time_step"]], test[["txId", "time_step"]]], ignore_index=True)
            .drop_duplicates("txId").reset_index(drop=True))
graph_table = graph_features(tx_steps, edges)
GRAPH_COLS = [c for c in graph_table.columns if c != "txId"]
BASE_COLS = FEAT_COLS + GRAPH_COLS
# add the graph features to the training and test tables
train_feats = train.merge(graph_table, on="txId", how="left")
test_feats = test.merge(graph_table, on="txId", how="left")

## 3. The model
`simple_model(fit_df, score_df)` trains on the labelled transactions in `fit_df` and scores `score_df`.

1. **How much each feature changed over time.** A classifier tries to tell `fit_df` rows from `score_df` rows using
   features only. The features it relies on most are the ones that drift.
2. **How useful each feature is.** How much a classifier that separates illicit from licit transactions relies on it.
3. **Rule.** Of the 30 features that changed most, those also in the top 60 for usefulness are converted to ranks
   and the rest are dropped. All other features are kept as they are.
4. **Percentile rank within its time step.** If a feature grows over time for every transaction, its raw value
   mostly tells you the date. Its rank within its own time step means the same thing in any period.
5. **Top 44.** Keep the 44 features a LightGBM model uses most. Fewer features give the model fewer chances to
   learn patterns that only held in the past.
6. **XGBoost, 10 seeds.** 800 trees, learning rate 0.02, depth 6, 60% of features per tree. Illicit transactions
   get about 7.6 times the weight, because only 11.7% of labelled transactions are illicit. Averaging 10 seeds
   smooths out run-to-run variation.

In [3]:
# Define the model: feature selection, then XGBoost averaged over 10 seeds.
SELECTION_PARAMS = dict(learning_rate=0.03, max_depth=4, num_leaves=31, min_child_samples=20,
                        reg_alpha=0.1, reg_lambda=5.0, subsample=1.0, colsample_bytree=0.9)
XGB_PARAMS = dict(learning_rate=0.02, max_depth=6, min_child_weight=5,
                  reg_alpha=0.0, reg_lambda=0.5, subsample=1.0, colsample_bytree=0.6)
SEEDS = [42, 1, 7, 123, 2024, 99, 555, 7777, 31, 88]  # 10 random seeds, averaged


def selection_rule(drift_rank, label_rank):
    """Of the 30 features that changed most over time, keep the useful ones as ranks and drop the rest."""
    if drift_rank <= 30:
        return "REPLACE_WITH_RANK" if label_rank <= 60 else "DROP"
    return "KEEP_RAW"


def add_step_ranks(df, cols):
    """Add each feature's percentile rank within its time step, as `<col>_rank`."""
    ranked = df.groupby("time_step")[cols].rank(pct=True)
    ranked.columns = [f"{c}_rank" for c in cols]
    return pd.concat([df, ranked], axis=1)


def simple_model(fit_df, score_df):
    """Select 44 features using fit_df (with labels) and score_df (features only), then train XGBoost and score score_df."""
    # 1. change over time: can a model tell fit rows from score rows with each feature? (no labels)
    X_adv = pd.concat([fit_df[BASE_COLS], score_df[BASE_COLS]], ignore_index=True)
    y_adv = np.array([0] * len(fit_df) + [1] * len(score_df))
    drift_score = np.zeros(len(BASE_COLS))
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    for fit_idx, _ in skf.split(X_adv, y_adv):
        m = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, max_depth=6,
                               random_state=SEED, n_jobs=-1, verbosity=-1)
        m.fit(X_adv.iloc[fit_idx], y_adv[fit_idx])
        drift_score += m.feature_importances_ / skf.n_splits

    # 2. usefulness: how much does an illicit-vs-licit model rely on each feature?
    labelled = fit_df[fit_df["label"].notna()].copy()
    labelled["label"] = labelled["label"].astype(int)
    spw = (labelled["label"] == 0).sum() / (labelled["label"] == 1).sum()
    m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.03, max_depth=6, scale_pos_weight=spw,
                           random_state=SEED, n_jobs=-1, verbosity=-1)
    m.fit(labelled[BASE_COLS], labelled["label"])

    decisions = pd.DataFrame({"feature": BASE_COLS, "drift_score": drift_score,
                              "label_score": m.feature_importances_})
    decisions["drift_rank"] = decisions["drift_score"].rank(ascending=False)
    decisions["label_rank"] = decisions["label_score"].rank(ascending=False)
    decisions["decision"] = [selection_rule(a, b) for a, b in zip(decisions.drift_rank, decisions.label_rank)]
    rank_cols = decisions.loc[decisions["decision"] == "REPLACE_WITH_RANK", "feature"].tolist()
    raw_cols = decisions.loc[decisions["decision"] == "KEEP_RAW", "feature"].tolist()

    # 3. percentile rank within its time step, for the features that drift but are useful
    fit_ranked = add_step_ranks(fit_df, rank_cols)
    score_ranked = add_step_ranks(score_df, rank_cols)
    candidate_cols = raw_cols + [f"{c}_rank" for c in rank_cols]
    labelled = fit_ranked[fit_ranked["label"].notna()].copy()
    labelled["label"] = labelled["label"].astype(int)
    y = labelled["label"]

    # 4. keep the 44 features a LightGBM model uses most
    m = lgb.LGBMClassifier(n_estimators=500, scale_pos_weight=0.5 * (y == 0).sum() / (y == 1).sum(),
                           random_state=SEED, n_jobs=-1, verbosity=-1, **SELECTION_PARAMS)
    m.fit(labelled[candidate_cols], y)
    top44 = pd.Series(m.feature_importances_, index=candidate_cols).sort_values(ascending=False).head(44).index.tolist()

    # 5. XGBoost on the 44 features, averaged over 10 seeds; illicit rows weighted by licit/illicit ratio
    spw = (y == 0).sum() / (y == 1).sum()
    seed_scores = []
    for seed in SEEDS:
        model = xgb.XGBClassifier(n_estimators=800, scale_pos_weight=spw, random_state=seed, n_jobs=-1,
                                  eval_metric="auc", verbosity=0, tree_method="hist", **XGB_PARAMS)
        model.fit(labelled[top44], y)
        seed_scores.append(model.predict_proba(score_ranked[top44])[:, 1])
    return decisions, top44, np.mean(seed_scores, axis=0)

## 4. Train on steps 1–35 and write the submission
Here the drift score compares training transactions (steps 1–35) with test transactions (36–49), using features
only. The file has 15,329 rows and the columns `index,target`, in the same order as `sample_submission.csv`.

In [4]:
# Train on steps 1-35, score the test transactions and write the submission file.
decisions, top44, final_pred = simple_model(train_feats, test_feats)
print("feature decisions:", decisions["decision"].value_counts().to_dict())
print("top-44 features:", top44)

# the file must match the sample submission row for row
sub = pd.DataFrame({"index": test["index"], "target": final_pred})
sample = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert len(sub) == len(sample) and (sub["index"].to_numpy() == sample["index"].to_numpy()).all()
assert sub["target"].between(0, 1).all() and sub["target"].notna().all()
out = OUT_DIR / "submission_2_e1122390.csv"
sub.to_csv(out, index=False)
print(f"wrote {out.relative_to(ROOT)} ({len(sub)} rows)")

feature decisions: {'KEEP_RAW': 143, 'DROP': 16, 'REPLACE_WITH_RANK': 13}
top-44 features: ['feat_53', 'feat_2_rank', 'feat_29', 'feat_59', 'feat_132', 'feat_3', 'feat_125', 'feat_46', 'feat_90', 'feat_52', 'feat_61', 'feat_138', 'feat_131', 'feat_161', 'feat_106', 'feat_5_rank', 'feat_8_rank', 'feat_55_rank', 'feat_142', 'feat_47', 'feat_127', 'feat_16_rank', 'feat_154', 'feat_89', 'feat_163', 'feat_88', 'feat_11_rank', 'feat_80', 'feat_143', 'feat_76_rank', 'feat_58_rank', 'feat_41', 'feat_18_rank', 'feat_102', 'feat_49', 'feat_164', 'feat_107', 'feat_136', 'feat_77', 'feat_65', 'feat_95', 'feat_94', 'feat_137', 'feat_155']
wrote Submission/submission_2_e1122390.csv (15329 rows)


## 5. Time-based validation: ROC-AUC, precision, recall and F1
The test labels are hidden, so I measure these on training steps the model has not seen. The same model trains on
steps 1–21 only and scores steps 22–35, whose labels stay hidden until scoring.

ROC-AUC needs no cut-off: it is the chance that a random illicit transaction scores above a random licit one.
Precision, recall and F1 need a yes or no answer, so I flag the top-scored transactions as illicit. In training
steps 1–21, 9.6% of labelled transactions are illicit, so the main cut-off flags the top 9.6%. It is fixed before
the validation steps are scored, so it uses none of their labels. The other rows show how precision and recall
change when more or fewer transactions are flagged.

In [5]:
# Time-based validation: train on steps 1-21, evaluate on 22-35, then report ROC-AUC, precision,
# recall and F1.
if RUN_TIME_CHECK:
    # the same model, trained on steps 1-21 only; the labels of steps 22-35 stay hidden
    early = train_feats[train_feats.time_step <= 21].reset_index(drop=True)
    late = train_feats[train_feats.time_step.between(22, 35)].reset_index(drop=True)
    _, _, check_score = simple_model(early, late.drop(columns="label"))

    labelled = late.label.notna().to_numpy()
    y_lab, s_lab = late.label[labelled].astype(int).to_numpy(), check_score[labelled]
    # main cut-off fixed before scoring: the illicit share of the labelled training rows (steps 1-21)
    train_share = early.label.mean()
    print(f"ROC-AUC (no cut-off needed): {roc_auc_score(y_lab, s_lab):.4f}")
    print(f"illicit share of labelled training rows (steps 1-21): {train_share:.1%} -> main cut-off")
    rows = []
    for share in (train_share, 0.05, 0.10, 0.15):
        flag = s_lab >= np.quantile(s_lab, 1 - share)  # flag the top `share` of transactions as illicit
        rows.append({"flagged": f"top {share:.1%}" + (" (main)" if share == train_share else ""),
                     "n flagged": int(flag.sum()), "illicit among flagged": int((flag & (y_lab == 1)).sum()),
                     "precision": precision_score(y_lab, flag), "recall": recall_score(y_lab, flag),
                     "F1": f1_score(y_lab, flag)})
    display(pd.DataFrame(rows).set_index("flagged").round(4))

ROC-AUC (no cut-off needed): 0.9056
illicit share of labelled training rows (steps 1-21): 9.6% -> main cut-off


,n flagged,illicit among flagged,precision,recall,F1
flagged,,,,,
top 9.6% (main),1122,1112,0.9911,0.6272,0.7682
top 5.0%,586,586,1.0000,0.3305,0.4968
top 10.0%,1171,1156,0.9872,0.6520,0.7853
top 15.0%,1756,1303,0.7420,0.7349,0.7385
